In [21]:
import pandas as pd
import numpy as np
import re

# Load cleaned fraud data and IP mapping
df_fraud = pd.read_csv('../data/processed/fraud_data_cleaned.csv')
ip_map = pd.read_csv('../data/processed/ip_country_cleaned.csv')

print("Fraud data shape:", df_fraud.shape)
print("IP map shape:", ip_map.shape)

# Ensure ip_address is string and filter valid IPv4
df_fraud['ip_address'] = df_fraud['ip_address'].astype(str)
ip_pattern = r'^\d+\.\d+\.\d+\.\d+$'
df_fraud = df_fraud[df_fraud['ip_address'].str.match(ip_pattern, na=False)]

def ip_to_int(ip):
    parts = ip.split('.')
    return int(parts[0]) * 256**3 + int(parts[1]) * 256**2 + int(parts[2]) * 256 + int(parts[3])

df_fraud['ip_int'] = df_fraud['ip_address'].apply(ip_to_int)
print("IP to integer conversion done.")

Fraud data shape: (151112, 11)
IP map shape: (138846, 3)
IP to integer conversion done.


In [22]:
# Convert IP range columns to integers
ip_map['lower'] = pd.to_numeric(ip_map['lower_bound_ip_address'], errors='coerce').astype('Int64')
ip_map['upper'] = pd.to_numeric(ip_map['upper_bound_ip_address'], errors='coerce').astype('Int64')
# Drop rows where conversion failed (should be none)
ip_map = ip_map.dropna(subset=['lower', 'upper'])

# Create an interval index (closed on both ends)
intervals = pd.IntervalIndex.from_arrays(ip_map['lower'], ip_map['upper'], closed='both')
print("Interval index created with", len(intervals), "ranges.")

Interval index created with 138846 ranges.


In [23]:
def get_country(ip_int):
    # Find the index of the interval containing ip_int
    idx = intervals.get_indexer([ip_int])[0]
    if idx == -1:
        return 'Unknown'
    else:
        return ip_map.iloc[idx]['country']

df_fraud['country'] = df_fraud['ip_int'].apply(get_country)
print("Country assignment complete.")

Country assignment complete.


In [24]:
# Top 10 countries by fraud rate
fraud_by_country = df_fraud.groupby('country')['class'].mean().sort_values(ascending=False)
print("Fraud rate by country (top 10):")
print(fraud_by_country.head(10))

# Save enriched dataset
df_fraud.to_csv('../data/processed/fraud_data_enriched.csv', index=False)
print("Saved enriched fraud data with country information.")

Fraud rate by country (top 10):
Series([], Name: class, dtype: float64)
Saved enriched fraud data with country information.
